This notebook extends the ["Parallel Execution (Part 2)"](./Parallel%20Execution%20%28Part%202%29.ipynb) example.

In [ ]:
# Install LangGraph — includes the 'interrupt' mechanism for human-in-the-loop workflows
!pip install -q langgraph

In [ ]:
import operator
import time

from IPython.display import HTML, Image
from google.colab import userdata
from langchain_core.runnables import Runnable, RunnableConfig
from langgraph.checkpoint.base import BaseCheckpointSaver
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.types import Command, Interrupt, interrupt
# ^ interrupt(): pauses graph execution and waits for human input
# ^ Command:     used to resume a paused graph, optionally passing a value back
# ^ Interrupt:   the object returned when the graph pauses — contains the prompt shown to the user
from pathlib import Path
from typing import Annotated, TypedDict, List

# Helper: renders interrupt prompts as styled HTML boxes in Jupyter (easier to read than plain text)
def print_interrupts(interrupts: List[Interrupt]):
    for el in interrupts:
        display(HTML(f'<div style="border: 1px dashed red; margin: 5px; padding: 10px; white-space: pre-wrap;">{el.value}</div>'))

# Helper: renders the compiled graph as a PNG image
def display_graph(runnable: Runnable, output_png: Path) -> None:
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

# Helper: lists all checkpoints saved for a given thread
def explore_checkpoints(checkpointer: BaseCheckpointSaver, config: RunnableConfig):
    checkpoints = list(checkpointer.list(config))
    print(f"There are {len(checkpoints)} checkpoints in total:")
    for checkpoint in reversed(checkpoints):
        print(checkpoint)

# Helper: walks through the full state history step by step
def explore_state_history(compiled_state_graph: CompiledStateGraph, config: RunnableConfig):
    state_history = list(compiled_state_graph.get_state_history(config))

    for snapshot in reversed(state_history):
        print(f"Step: {snapshot.metadata['step']}")
        print("Current state:")
        print(snapshot.values)
        print(f"Next: {snapshot.next}")
        print()

In [ ]:
# Custom reducer for 'findings': merges two dicts, with 'b' winning on key conflicts
def merge_findings(a: dict[str, str], b: dict[str, str]) -> dict[str, str]:
    # `b` overwrites `a` in case of conflicts.
    # This fragment can be modified to raise an error instead.
    return {**a, **b}

# Extended research state — adds verification tracking fields on top of findings/summary
class ResearchState(TypedDict):
    topic: str
    findings: Annotated[dict[str, str], merge_findings]         # Merged results from all sources
    verification_requests: Annotated[list[str], operator.add]   # Sources that need human verification (accumulated)
    verified: Annotated[set[str], operator.or_]                 # Sources approved by the human (union-merged)
    blocked: Annotated[set[str], operator.or_]                  # Sources rejected by the human (union-merged)
    summary: str                                                 # Final summary (only includes verified sources)

In [ ]:
# --- Data-fetching nodes (run in parallel) ---

def wikipedia(state: ResearchState):
    print("[WIKIPEDIA] node is executing")
    time.sleep(2)  # Simulate a slow network call
    # 'verification_requests' accumulates — operator.add means LangGraph appends to the list
    return { "findings": { "wikipedia": f"[WIKIPEDIA] {state['topic']}" }, "verification_requests": ["wikipedia"] }

def news(state: ResearchState):
    print("[LIVE NEWS] node is executing")
    time.sleep(2)
    return { "findings": { "news": f"[LIVE NEWS] Sensational! {state['topic']}" }, "verification_requests": ["news"] }

def arxiv(state: ResearchState):
    print("[ARXIV] node is executing")
    time.sleep(2)
    # ArXiv results don't need verification — no 'verification_requests' entry
    return { "findings": { "arxiv": f"[ARXIV] The theoretical hyperchaotic entanglement of relativities related to \"{state['topic']}\"" } }

# --- Human-in-the-loop verifier node ---
def verify(state: ResearchState):
    # This node handles each pending verification request ONE AT A TIME.
    # For each source that hasn't been decided on yet, it calls interrupt() to pause the graph
    # and wait for a human to approve (True) or reject (False) the source.
    current_verified = state.get('verified', set())
    current_blocked = state.get('blocked', set())

    new_verified = set()
    new_blocked = set()

    for request in state.get('verification_requests', {}):
        # Skip sources that were already decided in a previous resume cycle
        if request in current_verified or request in current_blocked:
            continue

        # interrupt() pauses the graph here and surfaces the message to the user.
        # Execution will resume from this exact point when the user calls graph.invoke(Command(resume=...))
        decision = interrupt(f"Pending verification:\n{state['findings'][request]}")

        # The human's decision (True = approve, False = reject) is returned by interrupt()
        if decision:
            new_verified.add(request)
        else:
            new_blocked.add(request)

    return { "verified": new_verified, "blocked": new_blocked }

def _annotate_finding(finding_key: str, state: ResearchState):
    # Helper: adds a "(verified)" or "(blocked)" label to each finding in the summary
    if finding_key in state.get('verified', set()):
        return " (verified)"
    elif finding_key in state.get('blocked', set()):
        return " (blocked)"
    else:
        return ""

def merge(state: ResearchState):
    # Only runs after ALL verification requests have been resolved.
    # If there are still pending verifications, skip and wait for the next resume cycle.
    if len(state.get('verification_requests', [])) > len(state.get('verified', set())) + len(state.get('blocked', set())):
        print("There are pending verification requests. Merge node will be skipped.")
        return {}

    # Build the summary including verification labels for each finding
    bullets = "\n".join(f" - {finding}{_annotate_finding(key, state)}" for key, finding in state.get("findings", {}).items())
    return { "summary": f"Summary:\n{bullets}" }

In [ ]:
in_memory_checkpointer = InMemorySaver()

In [ ]:
# Build the graph with parallel data fetching + human verification + merge.
#
# Flow:
#   START -> [wikipedia, news, arxiv] (parallel)
#         -> verifier (pauses for each source needing human approval)
#         -> merge    (only runs once all verifications are done)
#         -> END
graph_builder = StateGraph(ResearchState)
graph_builder.add_node("wikipedia", wikipedia)
graph_builder.add_node("news", news)
graph_builder.add_node("arxiv", arxiv)
graph_builder.add_node("verifier", verify)
graph_builder.add_node("merge", merge)

# NOTE: The three nodes ("wikipedia", "news", "arxiv") will be executed in parallel, then two of them will be verified and only after that they will be merged together.
graph_builder.add_edge(START, "wikipedia")
graph_builder.add_edge(START, "news")
graph_builder.add_edge(START, "arxiv")

# Wikipedia and news both go through the verifier; arxiv goes directly to merge
graph_builder.add_edge("wikipedia", "verifier")
graph_builder.add_edge("news", "verifier")
graph_builder.add_edge("verifier", "merge")
graph_builder.add_edge("arxiv", "merge")

graph_builder.add_edge("merge", END)

# A checkpointer is necessary in order to resume the execution later.
# Without it, the graph cannot save state when interrupted and cannot be resumed.
graph = graph_builder.compile(checkpointer=in_memory_checkpointer)

In [ ]:
# Visualize the graph structure — note the parallel fan-out and the verifier before merge
display_graph(graph, Path("/content/graph.png"))

In [ ]:
thread1_config = { "configurable": { "thread_id": "thread_3" } }
# Run the graph — it will pause at the FIRST interrupt (verification of wikipedia or news)
# and return the interrupt details instead of a final result
thread1_result = graph.invoke(input={ "topic": "Maximum snooker break" }, config=thread1_config)

In [ ]:
# Display the pending verification prompt — this is what the human should read and decide on
print_interrupts(thread1_result['__interrupt__'])

In [ ]:
# NOTE: Look at the pending writes for the last checkpoint.
# You'll see the graph is suspended mid-execution, waiting for input.
explore_checkpoints(in_memory_checkpointer, thread1_config)

In [ ]:
# Resume the graph and APPROVE the first source (True = verified).
# Command(resume=True) sends True as the return value of the interrupted interrupt() call.
# The graph picks up exactly where it left off and moves to the next verification.
thread1_result = graph.invoke(input=Command(resume=True), config=thread1_config)

In [ ]:
# Display the SECOND verification prompt — this will be for the 'news' source
print_interrupts(thread1_result['__interrupt__'])

In [ ]:
# NOTE: Look at the pending writes for the last checkpoint.
# The first source is now verified; the second is still pending.
explore_checkpoints(in_memory_checkpointer, thread1_config)

In [ ]:
# Resume the graph and REJECT the second source (False = blocked).
# After this, all verifications are resolved and the graph will continue to 'merge' and END.
thread1_result = graph.invoke(input=Command(resume=False), config=thread1_config)

In [ ]:
# Print the final summary — it will show "(verified)" and "(blocked)" labels next to each source
print(thread1_result['summary'])

In [ ]:
# Inspect checkpoints after the full run — the graph is now complete
explore_checkpoints(in_memory_checkpointer, thread1_config)

In [ ]:
# Walk through the entire state history — includes the paused steps where interrupts occurred
explore_state_history(graph, thread1_config)